In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report


In [ ]:
DATA_FILE = "production data with category.xlsx"
df = pd.read_excel(DATA_FILE)

print(df.shape)
df.head()


In [ ]:
# Smart if demand met or intentionally overproduced
df["Smart_Decision"] = (df["Production Gap"] <= 0).astype(int)

df["Smart_Decision"].value_counts()


In [ ]:
# Days of demand this plan covers
df["Days_Covered"] = df["Required Qty"] / df["Daily Requirement"]

# Parts that need daily attention are painful
df["Needs_Daily_Attention"] = (df["Days_Covered"] < 1.5).astype(int)


In [ ]:
# Category
df["Category_enc"] = df["Category"].map({
    "Runner": 0,
    "Repeater": 1,
    "Stranger": 2
})

# Shift (MAKE SURE COLUMN NAME IS CORRECT)
df["Shift_enc"] = df["Shift"].map({"A": 0, "B": 1})

# Machine
le_machine = LabelEncoder()
df["Machine_enc"] = le_machine.fit_transform(df["Machine ID"])


In [ ]:
def parse_cycle(x):
    if isinstance(x, str) and "/" in x:
        a, b = x.split("/")
        return float(a) / float(b)
    return np.nan

df["STD_Cycle"] = df["STD. Cycle Time"].apply(parse_cycle)
df["ACT_Cycle"] = df["Actual Standard Cycle Time"].apply(parse_cycle)


In [ ]:
FEATURES = [
    "Machine_enc",
    "Category_enc",
    "Shift_enc",
    "Required Qty",
    "Daily Requirement",
    "Days_Covered",
    "Needs_Daily_Attention",
    "Part Per Hour",
    "Hours Required",
    "Setup Rejections",
    "Actual Rejections",
    "STD_Cycle",
    "ACT_Cycle"
]

X = df[FEATURES].fillna(0)
y = df["Smart_Decision"]


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)


In [ ]:
model = RandomForestClassifier(
    n_estimators=200,
    max_depth=8,
    random_state=42,
    class_weight="balanced"
)

model.fit(X_train, y_train)
    

In [ ]:
y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


In [ ]:
machine_risk = (
    df.groupby("Machine ID")[["Setup Rejections", "Actual Rejections"]]
    .mean()
    .reset_index()
)


In [ ]:
parts_today = pd.DataFrame([
    {
        "Part": "P1",
        "Category": "Stranger",
        "Planned_Qty": 300,
        "Daily Requirement": 50,
        "Part Per Hour": 50,
        "STD_Cycle": 36,
        "ACT_Cycle": 34,
        "Eligible_Machines": ["MP-01", "MP-05"]
    },
    {
        "Part": "P2",
        "Category": "Repeater",
        "Planned_Qty": 250,
        "Daily Requirement": 250,
        "Part Per Hour": 80,
        "STD_Cycle": 28,
        "ACT_Cycle": 27,
        "Eligible_Machines": ["MP-01", "MP-10"]
    }
])


In [ ]:
candidates = []

for _, part in parts_today.iterrows():
    for m in part["Eligible_Machines"]:

        risk_row = machine_risk[machine_risk["Machine ID"] == m]

        if len(risk_row) == 0:
            setup_rej = 0
            actual_rej = 0
        else:
            setup_rej = risk_row["Setup Rejections"].values[0]
            actual_rej = risk_row["Actual Rejections"].values[0]

        days_covered = part["Planned_Qty"] / part["Daily Requirement"]

        candidates.append({
            "Part": part["Part"],
            "Machine ID": m,
            "Category": part["Category"],
            "Shift": "A",
            "Required Qty": part["Planned_Qty"],
            "Daily Requirement": part["Daily Requirement"],
            "Days_Covered": days_covered,
            "Needs_Daily_Attention": int(days_covered < 1.5),
            "Part Per Hour": part["Part Per Hour"],
            "Hours Required": part["Planned_Qty"] / part["Part Per Hour"],
            "Setup Rejections": setup_rej,
            "Actual Rejections": actual_rej,
            "STD_Cycle": part["STD_Cycle"],
            "ACT_Cycle": part["ACT_Cycle"]
        })

candidates = pd.DataFrame(candidates)


In [ ]:
candidates["Category_enc"] = candidates["Category"].map({
    "Runner": 0,
    "Repeater": 1,
    "Stranger": 2
})

candidates["Shift_enc"] = candidates["Shift"].map({"A": 0, "B": 1})
candidates["Machine_enc"] = le_machine.transform(candidates["Machine ID"])

X_candidates = candidates[FEATURES].fillna(0)


In [ ]:
candidates["Smart_Score"] = model.predict_proba(X_candidates)[:, 1]
candidates = candidates.sort_values("Smart_Score", ascending=False)

candidates[["Part", "Machine ID", "Days_Covered", "Smart_Score"]]


In [ ]:
MAX_HOURS = 22
CHANGEOVER = 40 / 60
MAX_PARTS = 4

machine_hours = {}
machine_parts = {}

schedule = []

for _, row in candidates.iterrows():

    m = row["Machine ID"]
    hrs = row["Hours Required"]

    machine_hours.setdefault(m, 0)
    machine_parts.setdefault(m, 0)

    if row["Smart_Score"] < 0.6:
        continue

    if machine_hours[m] + hrs + CHANGEOVER > MAX_HOURS:
        continue

    if machine_parts[m] >= MAX_PARTS:
        continue

    machine_hours[m] += hrs + CHANGEOVER
    machine_parts[m] += 1

    schedule.append({
        "Machine": m,
        "Part": row["Part"],
        "Qty": row["Required Qty"],
        "Days_Covered": round(row["Days_Covered"], 2),
        "Smart_Score": round(row["Smart_Score"], 3)
    })


In [ ]:
final_plan = pd.DataFrame(schedule)
final_plan
